Cell 1: Establish Connection and View Tables

In [ ]:
import duckdb
import pandas as pd

# Connect in read-only mode using the database name from your explorer
con = duckdb.connect("backend/maritime_case12days.duckdb", read_only=True)

# List all tables in the database
display(con.execute("SHOW TABLES;").df())

IOException: IO Error: Cannot open file "C:\Users\ryder\OneDrive\Documents\Git Repositories\caseathon2026\backend\database\backend\maritime_case12days.duckdb": The system cannot find the path specified.


Cell 2: Inspect the Clean Baseline Data

In [ ]:
# View the pristine, untouched reference AIS data
df_raw = con.execute("""
    SELECT mmsi, voyage_id, base_datetime, lat, lon, sog, cog, vessel_type, window_id 
    FROM raw_reference_ais 
    LIMIT 10;
""").df()

print("Raw Reference AIS (Clean Baseline):")
display(df_raw)

Cell 3: Inspect the Degraded Streams

In [ ]:
# View the experimental streams, filtering for intentionally poisoned records
df_injected = con.execute("""
    SELECT mmsi, voyage_id, timestamp, lat, lon, sog, cog, vessel_type, window_id, source_tier, condition, is_injected_defect
    FROM injected_streams 
    WHERE is_injected_defect = TRUE
    LIMIT 10;
""").df()

print("Injected Streams (Poisoned Records):")
display(df_injected)

Cell 4: Compare Defect Rates Across Windows

In [ ]:
# Verify that Window 1 has 0% defects and defects scale up in Windows 2 through 6
df_progression = con.execute("""
    SELECT 
        condition,
        window_id,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_injected_defect THEN 1 ELSE 0 END) AS defective_records,
        ROUND(SUM(CASE WHEN is_injected_defect THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS defect_pct
    FROM injected_streams
    WHERE source_tier = 'TIER2_FAST'
    GROUP BY condition, window_id
    ORDER BY condition, window_id;
""").df()

print("Defect Progression by Window:")
display(df_progression)

Cell 5: Safely Close the Connection

In [ ]:
# Release the read lock on the database
con.close()